# Next Word Prediction using LSTM

## Project Description

This project builds a **Next Word Prediction Model** using Deep Learning.
Given a sequence of words, the model predicts the most probable next word — similar to how smartphone keyboards suggest the next word while typing.

The model is trained on a text dataset (Sherlock Holmes novel) and uses an **LSTM (Long Short-Term Memory)** neural network to learn language patterns.

---

##  Features

* Predicts next word(s) based on input text
* Uses LSTM for sequence learning
* Trained on real-world text dataset
* Memory-optimized for Google Colab
* Beginner-friendly implementation

---

##  Tech Stack

* **Python**
* **TensorFlow / Keras**
* **NumPy**
* **Google Colab**

---

##  How It Works

The project follows these steps:

1. **Dataset Loading**

   * Text file is loaded and converted to lowercase

2. **Text Cleaning**

   * Punctuation is removed

3. **Tokenization**

   * Words are converted into numeric sequences

4. **Sequence Creation**

   * Input-output pairs are generated using n-grams

5. **Padding**

   * Sequences are padded to equal length

6. **Model Training**

   * LSTM model learns word patterns

7. **Prediction**

   * Given a sentence, the model predicts next word(s)

---

##  Installation & Setup

### 1. Open Google Colab

Go to: https://colab.research.google.com/

### 2. Upload Dataset

Upload the file:

```
1661-0.txt
```

### 3. Install Dependencies

```bash
pip install tensorflow
```

### 4. Run All Cells

Execute the notebook step-by-step.

---

##  Usage

Example:

```
Input:  sherlock holmes
Output: sherlock holmes was a very
```

You can change the input text and number of predicted words in the code.

---

##  Code Structure

* **Data Preprocessing**
  Loads dataset, cleans text, and tokenizes words

* **Sequence Generation**
  Creates training sequences

* **Model Building**
  Embedding + LSTM + Dense layers

* **Training**
  Model learns patterns from text

* **Prediction Function**
  Generates next words

---

##  Optimization Notes

To prevent memory crashes in Google Colab:

* Dataset size was limited
* Vocabulary size reduced (`num_words=5000`)
* Used `sparse_categorical_crossentropy` instead of one-hot encoding
* Reduced epochs and model size

These optimizations ensure smooth training even on limited RAM.

---

##  Future Improvements

* Use **GRU or Transformer models**
* Build a **web interface (Streamlit/Flask)**
* Deploy on **AWS (EC2 / Lambda)**
* Train on larger datasets for better accuracy


---

## Author

* Mr Lfee Deshwal

---

## Final Note

This project is a great starting point for understanding:

* Natural Language Processing (NLP)
* Sequence Models
* Deep Learning with LSTM

Feel free to improve and extend it


In [ ]:
import numpy as np
import string
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

print("Imports consolidated successfully.")

Imports consolidated successfully.


In [ ]:
import numpy as np
import string
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# 1. Read the text file and convert to lowercase
with open('/content/1661-0.txt', 'r', encoding='utf-8') as file:
    data = file.read().lower()[:50000]

# 2. Clean the text by removing punctuation
cleaned_data = data.translate(str.maketrans('', '', string.punctuation))

# 3. Initialize Tokenizer and fit on text (Limit num_words to 5000)
tokenizer = Tokenizer(num_words=5000)
tokenizer.fit_on_texts([cleaned_data])
total_words = len(tokenizer.word_index) + 1

# 4. Create n-gram input sequences (Process only the first 2000 lines)
input_sequences = []
for line in cleaned_data.split('\n')[:2000]:
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        n_gram_seq = token_list[:i+1]
        input_sequences.append(n_gram_seq)

# 5. Determine max sequence length and pad sequences
max_seq_len = max([len(x) for x in input_sequences]) if input_sequences else 0
input_sequences = np.array(pad_sequences(input_sequences, maxlen=max_seq_len, padding='pre'))

# 6. Split sequences into predictors (X) and label (y)
X = input_sequences[:, :-1]
y = input_sequences[:, -1]

# 7. One-hot encode the labels (Removed to save memory)
# y = to_categorical(y, num_classes=total_words)

print(f'Total unique words: {total_words}')
print(f'Max sequence length: {max_seq_len}')
print(f'X shape: {X.shape}')
print(f'y shape: {y.shape}')

Total unique words: 2310
Max sequence length: 18
X shape: (8241, 17)
y shape: (8241,)


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
import tensorflow as tf

# 1. Define a sequential model
model = Sequential()

# 2. Add an Embedding layer - using Input layer to ensure the model is built
model.add(tf.keras.Input(shape=(max_seq_len - 1,)))
model.add(Embedding(input_dim=total_words, output_dim=100))

# 3. Add an LSTM layer with 100 units
model.add(LSTM(100))

# 4. Add a Dense output layer with softmax activation
model.add(Dense(total_words, activation='softmax'))

# 5. Compile the model
model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

# 6. Verify the model structure
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 17, 100)        │       231,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 100)            │        80,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 2310)           │       233,310 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 544,710 (2.08 MB)

 Trainable params: 544,710 (2.08 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
import numpy as np
from tensorflow.keras.preprocessing.sequence import pad_sequences

# 1. Train the compiled model
# Using 5 epochs to reduce training time and memory usage for a test run
history = model.fit(X, y, epochs=5, verbose=1)

# 2. Define the prediction function
def predict_next_word(seed_text, next_words):
    for _ in range(next_words):
        # Convert seed_text to sequence of tokens
        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        # Pad the sequence
        token_list = pad_sequences([token_list], maxlen=max_seq_len-1, padding='pre')

        # Get probabilities and find the index of the highest probability
        predicted_probs = model.predict(token_list, verbose=0)
        predicted_index = np.argmax(predicted_probs, axis=-1)[0]

        # Map index back to word
        output_word = ""
        for word, index in tokenizer.word_index.items():
            if index == predicted_index:
                output_word = word
                break

        # Append predicted word to seed text
        seed_text += " " + output_word

    return seed_text

# 5. Call the function and print the result
seed = "sherlock holmes"
num_predicted_words = 5
result = predict_next_word(seed, num_predicted_words)
print(f'\nSeed text: {seed}')
print(f'Predicted text: {result}')

Epoch 1/5
258/258 ━━━━━━━━━━━━━━━━━━━━ 8s 24ms/step - accuracy: 0.0536 - loss: 6.5650
Epoch 2/5
258/258 ━━━━━━━━━━━━━━━━━━━━ 7s 26ms/step - accuracy: 0.0597 - loss: 6.0576
Epoch 3/5
258/258 ━━━━━━━━━━━━━━━━━━━━ 6s 23ms/step - accuracy: 0.0686 - loss: 5.9292
Epoch 4/5
258/258 ━━━━━━━━━━━━━━━━━━━━ 7s 27ms/step - accuracy: 0.0737 - loss: 5.8097
Epoch 5/5
258/258 ━━━━━━━━━━━━━━━━━━━━ 6s 22ms/step - accuracy: 0.0818 - loss: 5.6763

Seed text: sherlock holmes
Predicted text: sherlock holmes the king of the king


## Model Performance Note

The current implementation of this project is intentionally optimized to run efficiently on **Google Colab's limited memory environment**.

To prevent runtime crashes and ensure smooth execution, the following optimizations were applied:

* Reduced dataset size
* Limited vocabulary (`num_words=5000`)
* Restricted number of training sequences
* Used `sparse_categorical_crossentropy` instead of one-hot encoding
* Reduced model complexity and training epochs

###  Impact on Accuracy

Due to these constraints, the model may show **lower prediction accuracy and less coherent outputs** compared to a fully trained large-scale model.

###  Expected Performance with Scaling

If trained with:

* Full dataset-:https://drive.google.com/file/d/1GeUzNVqiixXHnTl8oNiQ2W3CynX_lsu2/view?usp=sharing
* Larger vocabulary
* More epochs
* Increased model size (LSTM units)

 The model performance and prediction quality will **significantly improve**, producing more accurate and context-aware results.

###  Key Takeaway

This project demonstrates the **complete pipeline of an NLP sequence model**, focusing on:

* Correct implementation
* Efficient resource usage
* Scalability for real-world deployment

The current version is a **lightweight prototype**, not a fully optimized production model.
